In [24]:
"""Feature Engineering     

    알고보니까 dropout 적용안하고 

    bath도 total bath가 아닌 이상하게 적용하고 있었음
    """


'Feature Engineering     \n\n    알고보니까 dropout 적용안하고 \n\n    bath도 total bath가 아닌 이상하게 적용하고 있었음\n    '

In [25]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

#최초의 신경망 

class HousePriceMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc3(x)

        return x.squeeze(1)

print(train.shape)
print(test.shape)
print(train.head())

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  

In [26]:
#현재: X와 y 분리
y = train["SalePrice"]
x = train.drop(columns=["Id","SalePrice"])

test_id = test["Id"]
x_test = test.drop(columns=["Id"])

print("x크기",x.shape)
print("y크기",y.shape)
print("test크기",x_test.shape)

x크기 (1460, 79)
y크기 (1460,)
test크기 (1459, 79)


In [27]:
def add_features(data):
    data = data.copy()

    data["HouseAge"] = (
        data["YrSold"] - data["YearBuilt"]
    )

    data["TotalBath"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    return data

x = add_features(x)
x_test = add_features(x_test)

In [28]:
#MLP에 넣으려면 최종적으로 x가 모든 값이 숫자& 결측값이 없어야함!
#문자형 43개, 정수형 33개, 실수형 3개
#결측값 해석 수영장 1460중에 1453개 없음
print(x.dtypes.value_counts())
print(x.isnull().sum().sort_values(ascending=False).head(20))

str        43
int64      34
float64     4
Name: count, dtype: int64
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageType        81
GarageFinish      81
GarageCond        81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
MSSubClass         0
dtype: int64


In [29]:
#첫번째 베이스라인에서는 너무 복잡하게 하지말고
#범주형 결측값 -> Missing
#수치형 결측값 -> 중앙값

In [30]:
"""지금 바로 결측값을 채우면 안 되는 이유

먼저 train 데이터를 다시 학습용과 검증용으로 나눠야 해.

x
├── x_train: 모델이 공부할 문제
└── x_valid: 모델이 풀어볼 시험문제

y
├── y_train: 공부할 문제의 정답
└── y_valid: 시험문제의 정답

결측값을 먼저 전체 데이터의 중앙값으로 채우면 validation 
데이터의 정보가 training에 조금 들어갈 수 있어. 
이것을 데이터 누수라고 해.

집값은 너무 비싸기 때문에 log로 취해서 사용
"""



'지금 바로 결측값을 채우면 안 되는 이유\n\n먼저 train 데이터를 다시 학습용과 검증용으로 나눠야 해.\n\nx\n├── x_train: 모델이 공부할 문제\n└── x_valid: 모델이 풀어볼 시험문제\n\ny\n├── y_train: 공부할 문제의 정답\n└── y_valid: 시험문제의 정답\n\n결측값을 먼저 전체 데이터의 중앙값으로 채우면 validation \n데이터의 정보가 training에 조금 들어갈 수 있어. \n이것을 데이터 누수라고 해.\n\n집값은 너무 비싸기 때문에 log로 취해서 사용\n'

In [31]:
y_log = np.log1p(y)

from sklearn.model_selection import train_test_split

x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y_log,
    test_size=0.2,
    random_state=42
)

# 학습 데이터의 평균만 사용
target_mean = float(y_train.mean())

# 평균을 빼서 0 근처로 중심화
y_train_centered = (
    y_train - target_mean
)

y_valid_centered = (
    y_valid - target_mean
)

print("타깃 평균:", target_mean)
print(
    "중심화 전 y_train 평균:",
    y_train.mean()
)
print(
    "중심화 후 y_train 평균:",
    y_train_centered.mean()
)

타깃 평균: 12.030658310971573
중심화 전 y_train 평균: 12.030658310971573
중심화 후 y_train 평균: 4.258389683493751e-16


In [32]:
numeric_columns = x_train.select_dtypes(
    include=["int64","float64"]
).columns

categorical_columns = x_train.select_dtypes(
    include=["object"]
).columns

print("수치형:", len(numeric_columns))
print("범주형:", len(categorical_columns))

수치형: 38
범주형: 43


/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_31074/3797188013.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x_train.select_dtypes(


In [33]:
numeric_medians = x_train[numeric_columns].median()

x_train[numeric_columns] = (
    x_train[numeric_columns].fillna(numeric_medians)
)

x_valid[numeric_columns] = (
    x_valid[numeric_columns].fillna(numeric_medians)
)

x_test[numeric_columns] = (
    x_test[numeric_columns].fillna(numeric_medians)
)

x_train[categorical_columns] = (
    x_train[categorical_columns].fillna("Missing")
)

x_valid[categorical_columns] = (
    x_valid[categorical_columns].fillna("Missing")
)

x_test[categorical_columns] = (
    x_test[categorical_columns].fillna("Missing")
)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.compose import ColumnTransformer

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

# 집값 로그 변환
y_log = np.log1p(y)

kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 전체 train 데이터의 OOF 예측 저장
oof_predictions = np.zeros(
    len(x),
    dtype=np.float32
)

# 5개 모델의 test 예측 누적
test_log_predictions = np.zeros(
    len(x_test),
    dtype=np.float32
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    kfold.split(x),
    start=1
):
    print(f"\n========== Fold {fold} ==========")

    # 이번 Fold의 데이터
    x_train_fold = x.iloc[train_idx].copy()
    x_valid_fold = x.iloc[valid_idx].copy()

    y_train_fold = y_log.iloc[train_idx].copy()
    y_valid_fold = y_log.iloc[valid_idx].copy()

    x_test_fold = x_test.copy()

    # =========================
    # 타깃 중심화
    # =========================

    # 이번 Fold의 학습 정답 평균만 사용
    target_mean = float(
        y_train_fold.mean()
    )

    y_train_centered = (
        y_train_fold - target_mean
    )

    y_valid_centered = (
        y_valid_fold - target_mean
    )

    print("target 평균:", target_mean)
    print(
        "중심화 후 평균:",
        y_train_centered.mean()
    )

    # =========================
    # 수치형과 범주형 구분
    # =========================

    numeric_columns = (
        x_train_fold
        .select_dtypes(
            include=["int64", "float64"]
        )
        .columns
    )

    categorical_columns = (
        x_train_fold
        .select_dtypes(
            include=["object"]
        )
        .columns
    )

    # =========================
    # 결측값 처리
    # =========================

    # 이번 Fold의 train 중앙값만 계산
    numeric_medians = (
        x_train_fold[numeric_columns]
        .median()
    )

    for data in [
        x_train_fold,
        x_valid_fold,
        x_test_fold
    ]:
        data[numeric_columns] = (
            data[numeric_columns]
            .fillna(numeric_medians)
        )

        data[categorical_columns] = (
            data[categorical_columns]
            .fillna("Missing")
        )

    # =========================
    # 스케일링과 원핫인코딩
    # =========================

    # Fold마다 새로운 전처리기 생성
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                StandardScaler(),
                numeric_columns
            ),
            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_columns
            )
        ]
    )

    # train에만 fit
    x_train_processed = (
        preprocessor.fit_transform(
            x_train_fold
        )
    )

    x_valid_processed = (
        preprocessor.transform(
            x_valid_fold
        )
    )

    x_test_processed = (
        preprocessor.transform(
            x_test_fold
        )
    )

    # =========================
    # Tensor 변환
    # =========================

    x_train_tensor = torch.tensor(
        x_train_processed,
        dtype=torch.float32
    )

    x_valid_tensor = torch.tensor(
        x_valid_processed,
        dtype=torch.float32
    )

    x_test_tensor = torch.tensor(
        x_test_processed,
        dtype=torch.float32
    )

    # 중심화한 정답 사용
    y_train_tensor = torch.tensor(
        y_train_centered.values,
        dtype=torch.float32
    )

    y_valid_tensor = torch.tensor(
        y_valid_centered.values,
        dtype=torch.float32
    )

    # =========================
    # DataLoader 생성
    # =========================

    train_dataset = TensorDataset(
        x_train_tensor,
        y_train_tensor
    )

    valid_dataset = TensorDataset(
        x_valid_tensor,
        y_valid_tensor
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=32,
        shuffle=False
    )

    # =========================
    # Fold마다 새로운 모델
    # =========================

    torch.manual_seed(42 + fold)

    input_dim = x_train_tensor.shape[1]

    model = HousePriceMLP(input_dim)

    criterion = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=0.0001
    )

    best_valid_rmse = float("inf")
    patience = 20
    patience_counter = 0

    checkpoint_path = (
        f"best_house_mlp_fold_{fold}.pt"
    )

    # =========================
    # 직접 작성한 학습 loop
    # =========================

    for epoch in range(200):

        model.train()

        total_train_loss = 0

        for batch_x, batch_y in train_loader:

            optimizer.zero_grad()

            predictions = model(batch_x)

            loss = criterion(
                predictions,
                batch_y
            )

            loss.backward()
            optimizer.step()

            total_train_loss += (
                loss.item()
                * batch_x.size(0)
            )

        average_train_loss = (
            total_train_loss
            / len(train_loader.dataset)
        )

        # =====================
        # Validation loop
        # =====================

        model.eval()

        total_valid_loss = 0

        with torch.no_grad():

            for batch_x, batch_y in valid_loader:

                predictions = model(batch_x)

                loss = criterion(
                    predictions,
                    batch_y
                )

                total_valid_loss += (
                    loss.item()
                    * batch_x.size(0)
                )

        average_valid_loss = (
            total_valid_loss
            / len(valid_loader.dataset)
        )

        train_rmse = (
            average_train_loss ** 0.5
        )

        valid_rmse = (
            average_valid_loss ** 0.5
        )

        print(
            f"Fold {fold} | "
            f"Epoch {epoch + 1}: "
            f"Train={train_rmse:.4f}, "
            f"Valid={valid_rmse:.4f}"
        )

        # =====================
        # Best checkpoint
        # =====================

        if valid_rmse < best_valid_rmse:

            best_valid_rmse = valid_rmse
            patience_counter = 0

            torch.save(
                model.state_dict(),
                checkpoint_path
            )

        else:
            patience_counter += 1

        # =====================
        # Early stopping
        # =====================

        if patience_counter >= patience:

            print(
                f"Fold {fold} Early Stopping!"
            )

            break

    # =========================
    # Best 모델 불러오기
    # =========================

    model.load_state_dict(
        torch.load(
            checkpoint_path,
            weights_only=True
        )
    )

    model.eval()

    with torch.no_grad():

        # 모델은 중심화된 값을 예측
        valid_centered_predictions = (
            model(x_valid_tensor)
            .cpu()
            .numpy()
        )

        test_centered_predictions = (
            model(x_test_tensor)
            .cpu()
            .numpy()
        )

    # Fold의 평균을 다시 더해서
    # 원래 로그 집값으로 복구
    valid_log_predictions = (
        valid_centered_predictions
        + target_mean
    )

    fold_test_log_predictions = (
        test_centered_predictions
        + target_mean
    )

    # OOF 예측을 원래 위치에 저장
    oof_predictions[valid_idx] = (
        valid_log_predictions
    )

    # 5개 Fold의 test 예측 평균
    test_log_predictions += (
        fold_test_log_predictions / 5
    )

    # 중심화하기 전 y_valid와 비교
    fold_rmse = np.sqrt(
        np.mean(
            (
                valid_log_predictions
                - y_valid_fold.values
            ) ** 2
        )
    )

    fold_scores.append(fold_rmse)

    print(
        f"Fold {fold} Best RMSE: "
        f"{fold_rmse:.5f}"
    )

# =============================
# 전체 OOF RMSE
# =============================

oof_rmse = np.sqrt(
    np.mean(
        (
            oof_predictions
            - y_log.values
        ) ** 2
    )
)

print("\nFold별 RMSE:", fold_scores)
print(
    "Fold 평균 RMSE:",
    np.mean(fold_scores)
)
print(
    "전체 OOF RMSE:",
    oof_rmse
)

# =============================
# 실제 집값으로 복구
# =============================

test_predictions = np.expm1(
    test_log_predictions
)

submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(
    "submission.csv "
    "생성 완료!"
)


========== Fold 1 ==========
target 평균: 12.030658310971573
중심화 후 평균: 4.258389683493751e-16
Fold 1 | Epoch 1: Train=0.2128, Valid=0.1540
Fold 1 | Epoch 2: Train=0.1479, Valid=0.1443
Fold 1 | Epoch 3: Train=0.1265, Valid=0.1333
Fold 1 | Epoch 4: Train=0.1226, Valid=0.1390
Fold 1 | Epoch 5: Train=0.1068, Valid=0.1375
Fold 1 | Epoch 6: Train=0.1065, Valid=0.1487
Fold 1 | Epoch 7: Train=0.0955, Valid=0.1365
Fold 1 | Epoch 8: Train=0.0899, Valid=0.1371


/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_31074/816957117.py:88: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


Fold 1 | Epoch 9: Train=0.0885, Valid=0.1354
Fold 1 | Epoch 10: Train=0.0832, Valid=0.1388
Fold 1 | Epoch 11: Train=0.0934, Valid=0.1431
Fold 1 | Epoch 12: Train=0.0783, Valid=0.1412
Fold 1 | Epoch 13: Train=0.0771, Valid=0.1373
Fold 1 | Epoch 14: Train=0.0758, Valid=0.1437
Fold 1 | Epoch 15: Train=0.0695, Valid=0.1370
Fold 1 | Epoch 16: Train=0.0691, Valid=0.1428
Fold 1 | Epoch 17: Train=0.0677, Valid=0.1469
Fold 1 | Epoch 18: Train=0.0683, Valid=0.1410
Fold 1 | Epoch 19: Train=0.0651, Valid=0.1391
Fold 1 | Epoch 20: Train=0.0671, Valid=0.1482
Fold 1 | Epoch 21: Train=0.0666, Valid=0.1491
Fold 1 | Epoch 22: Train=0.0660, Valid=0.1459
Fold 1 | Epoch 23: Train=0.0655, Valid=0.1403
Fold 1 Early Stopping!
Fold 1 Best RMSE: 0.13329

========== Fold 2 ==========
target 평균: 12.016898012594972
중심화 후 평균: -1.4965198030563755e-15
Fold 2 | Epoch 1: Train=0.2169, Valid=0.1350
Fold 2 | Epoch 2: Train=0.1465, Valid=0.1209
Fold 2 | Epoch 3: Train=0.1214, Valid=0.1310
Fold 2 | Epoch 4: Train=0.1165, V

/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_31074/816957117.py:88: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


Fold 2 | Epoch 12: Train=0.0799, Valid=0.1384
Fold 2 | Epoch 13: Train=0.0774, Valid=0.1297
Fold 2 | Epoch 14: Train=0.0787, Valid=0.1400
Fold 2 | Epoch 15: Train=0.0787, Valid=0.1287
Fold 2 | Epoch 16: Train=0.0742, Valid=0.1330
Fold 2 | Epoch 17: Train=0.0731, Valid=0.1318
Fold 2 | Epoch 18: Train=0.0677, Valid=0.1299
Fold 2 | Epoch 19: Train=0.0705, Valid=0.1296
Fold 2 | Epoch 20: Train=0.0663, Valid=0.1323
Fold 2 | Epoch 21: Train=0.0643, Valid=0.1366
Fold 2 | Epoch 22: Train=0.0650, Valid=0.1328
Fold 2 | Epoch 23: Train=0.0591, Valid=0.1294
Fold 2 | Epoch 24: Train=0.0632, Valid=0.1348
Fold 2 | Epoch 25: Train=0.0612, Valid=0.1343
Fold 2 | Epoch 26: Train=0.0586, Valid=0.1332
Fold 2 Early Stopping!
Fold 2 Best RMSE: 0.11911

========== Fold 3 ==========
target 평균: 12.022758503293897
중심화 후 평균: -3.7412995076409386e-16
Fold 3 | Epoch 1: Train=0.2051, Valid=0.2017
Fold 3 | Epoch 2: Train=0.1294, Valid=0.1848
Fold 3 | Epoch 3: Train=0.1202, Valid=0.1977
Fold 3 | Epoch 4: Train=0.1108, 

...

Fold 3 | Epoch 12: Train=0.0771, Valid=0.2167
Fold 3 | Epoch 13: Train=0.0766, Valid=0.2250
Fold 3 | Epoch 14: Train=0.0842, Valid=0.2263
Fold 3 | Epoch 15: Train=0.0732, Valid=0.2515
Fold 3 | Epoch 16: Train=0.0723, Valid=0.2367
Fold 3 | Epoch 17: Train=0.0729, Valid=0.2430
Fold 3 | Epoch 18: Train=0.0714, Valid=0.2470
Fold 3 | Epoch 19: Train=0.0659, Valid=0.2512
Fold 3 | Epoch 20: Train=0.0699, Valid=0.2423
Fold 3 | Epoch 21: Train=0.0613, Valid=0.2381
Fold 3 | Epoch 22: Train=0.0650, Valid=0.2480
Fold 3 Early Stopping!
Fold 3 Best RMSE: 0.18482

========== Fold 4 ==========
target 평균: 12.027932508834795
중심화 후 평균: -4.0454701993190635e-16
Fold 4 | Epoch 1: Train=0.2272, Valid=0.1423


/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_31074/816957117.py:88: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


Fold 4 | Epoch 2: Train=0.1453, Valid=0.1302
Fold 4 | Epoch 3: Train=0.1251, Valid=0.1259
Fold 4 | Epoch 4: Train=0.1241, Valid=0.1307
Fold 4 | Epoch 5: Train=0.1183, Valid=0.1475
Fold 4 | Epoch 6: Train=0.1092, Valid=0.1269
Fold 4 | Epoch 7: Train=0.0981, Valid=0.1320
Fold 4 | Epoch 8: Train=0.0968, Valid=0.1248
Fold 4 | Epoch 9: Train=0.0908, Valid=0.1264
Fold 4 | Epoch 10: Train=0.0865, Valid=0.1271
Fold 4 | Epoch 11: Train=0.0868, Valid=0.1324
Fold 4 | Epoch 12: Train=0.0828, Valid=0.1274
Fold 4 | Epoch 13: Train=0.0845, Valid=0.1302
Fold 4 | Epoch 14: Train=0.0809, Valid=0.1313
Fold 4 | Epoch 15: Train=0.0775, Valid=0.1284
Fold 4 | Epoch 16: Train=0.0759, Valid=0.1333
Fold 4 | Epoch 17: Train=0.0694, Valid=0.1285
Fold 4 | Epoch 18: Train=0.0744, Valid=0.1317
Fold 4 | Epoch 19: Train=0.0728, Valid=0.1340
Fold 4 | Epoch 20: Train=0.0684, Valid=0.1367
Fold 4 | Epoch 21: Train=0.0658, Valid=0.1308
Fold 4 | Epoch 22: Train=0.0689, Valid=0.1315
Fold 4 | Epoch 23: Train=0.0595, Valid=0.1

/var/folders/7x/1yqyfb3s3tn32qj1htyyynz80000gn/T/ipykernel_31074/816957117.py:88: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


Fold 5 | Epoch 1: Train=0.2310, Valid=0.1255
Fold 5 | Epoch 2: Train=0.1635, Valid=0.1212
Fold 5 | Epoch 3: Train=0.1381, Valid=0.1132
Fold 5 | Epoch 4: Train=0.1329, Valid=0.1116
Fold 5 | Epoch 5: Train=0.1161, Valid=0.1048
Fold 5 | Epoch 6: Train=0.1121, Valid=0.1064
Fold 5 | Epoch 7: Train=0.1100, Valid=0.1117
Fold 5 | Epoch 8: Train=0.0966, Valid=0.1136
Fold 5 | Epoch 9: Train=0.0941, Valid=0.1152
Fold 5 | Epoch 10: Train=0.0912, Valid=0.1231
Fold 5 | Epoch 11: Train=0.0957, Valid=0.1186
Fold 5 | Epoch 12: Train=0.0794, Valid=0.1070
Fold 5 | Epoch 13: Train=0.0856, Valid=0.1176
Fold 5 | Epoch 14: Train=0.0819, Valid=0.1078
Fold 5 | Epoch 15: Train=0.0746, Valid=0.1077
Fold 5 | Epoch 16: Train=0.0726, Valid=0.1177
Fold 5 | Epoch 17: Train=0.0789, Valid=0.1093
Fold 5 | Epoch 18: Train=0.0698, Valid=0.1125
Fold 5 | Epoch 19: Train=0.0695, Valid=0.1138
Fold 5 | Epoch 20: Train=0.0695, Valid=0.1110
Fold 5 | Epoch 21: Train=0.0677, Valid=0.1107
Fold 5 | Epoch 22: Train=0.0641, Valid=0.10